# Dataset Import Experiment

Author: Luca Cotti (<luca.cotti@unibs.it>)

This notebook contains an experiment for importing the "russellmitchell" dataset from the AIT-LDS repository.


## Setup

In [ ]:
import json
import os
import re
import shutil
from pathlib import Path

import pandas as pd
from thefuzz import fuzz

# Delete imported data if it already exists
DELETE_OUT_IF_EXISTS = True

# Path of the dataset to import
DATASET_PATH = "../../ait_dataset/russellmitchell"

# These log dir contain just the logs on user/attacker behavior generation
LOG_DIRS_TO_IGNORE = ["attacker", "ext_user", "internal_employee", "remote_employee"]

# Maximum number of lines to read from each log file
TRUNC_LINES = 10000

# Levenshtein similarity threshold for log comparisong
SIMILARITY_THRESHOLD = 90

# Where to store the processed csv files
OUT_DIR = "./out_full"

In [8]:
from typing import NamedTuple


class TTP(NamedTuple):
    """A MITRE ATT&CK TTP representation."""

    tactic: str
    techniques: list[str]


# Map the log files labels to TTPs, using the description provided in the paper
LABELS_TO_TTPS: dict[str, TTP] = {
    "['attacker_vpn', 'foothold']": TTP("Initial Access", ["Valid Accounts"]),
    "['service_scan', 'foothold']": TTP(
        "Reconnaissance",
        ["Active Scanning", "Gather Victim Network Information"],
    ),
    "['attacker_http', 'foothold', 'service_scan']": TTP(
        "Reconnaissance",
        ["Active Scanning", "Gather Victim Network Information"],
    ),
    "['traceroute', 'foothold']": TTP(
        "Reconnaissance",
        ["Active Scanning", "Gather Victim Network Information"],
    ),
    "['attacker_http', 'foothold', 'dirb']": TTP(
        "Reconnaissance",
        ["Active Scanning", "Gather Victim Host Information"],
    ),
    "['attacker_http', 'foothold', 'wpscan']": TTP(
        "Reconnaissance",
        ["Active Scanning", "Gather Victim Host Information"],
    ),
    "['attacker_http', 'foothold', 'webshell_upload']": TTP(
        "Execution",
        [
            "Exploitation for Client Execution Persistence",
            "Server Software Component Discovery",
        ],
    ),
    "['attacker_http', 'foothold', 'webshell_cmd']": TTP(
        "Persistence",
        ["Server Software Component Discovery"],
    ),
    "['webshell_cmd', 'escalate']": TTP("Credential Access", ["OS Credential Dumping"]),
    "['escalate', 'crack_passwords']": TTP(
        "Credential Access",
        ["Brute Force: Password Cracking"],
    ),
    "['attacker_change_user', 'escalate']": TTP("Privilege Escalation", ["Valid Accounts"]),
    "['attacker_change_user', 'escalate', 'escalated_command', 'escalated_sudo_command']": TTP(
        "Privilege Escalation",
        ["Valid Accounts"],
    ),
    "['escalated_command', 'escalated_sudo_command', 'escalate']": TTP(
        "Execution",
        ["Command and Scripting Interpreter"],
    ),
    "['escalated_command', 'escalated_sudo_command', 'escalate', 'escalated_sudo_session']": TTP(
        "Execution",
        ["Command and Scripting Interpreter"],
    ),
    "['escalated_command', 'escalated_sudo_command', 'escalated_sudo_session', 'escalate']": TTP(
        "Execution",
        ["Command and Scripting Interpreter"],
    ),
    "['dnsteal', 'exfiltration-service', 'attacker']": TTP(
        "Exfiltration",
        ["Exfiltration Over Alternative Protocol"],
    ),
    "['dnsteal', 'attacker', 'dnsteal-received']": TTP(
        "Exfiltration",
        ["Exfiltration Over Alternative Protocol"],
    ),
    "['dnsteal', 'attacker', 'dnsteal-dropped']": TTP(
        "Exfiltration",
        ["Exfiltration Over Alternative Protocol"],
    ),
}

## Load dataset

In [ ]:
# List all directories in the logs subdir of the DATASET_PATH
logs_subdir_path = Path(DATASET_PATH) / "gather"
logs_dirs = [d for d in Path.iterdir(logs_subdir_path) if Path.is_dir(d)]

labels_subdir_path = Path(DATASET_PATH) / "labels"

# Filter out directories containing "attacker", "ext_user", or "Internal_employee"
devices = [d.name for d in logs_dirs if not any(re.search(x, d.name) for x in LOG_DIRS_TO_IGNORE)]


def read_log_and_labels(device: str, app: str, file_name: str) -> pd.DataFrame:
    """Read a log file and returns a dataframe with the lines and line numbers."""
    file_path = Path(logs_subdir_path) / device / "logs" / app / file_name

    lines = []
    with Path.open(file_path, "r") as file:
        for i, line in enumerate(file):
            if i >= TRUNC_LINES:
                break

            lines.append({"line_number": i, "text": line.strip()})

    log_df = pd.DataFrame(lines)
    log_df["device"] = device
    log_df["app"] = app if app != "." else "system"
    log_df["file_name"] = file_name
    log_df["tactic"] = ""
    log_df["techniques"] = ""

    labels_path = labels_subdir_path / device / "logs" / app / file_name

    if Path.exists(labels_path):
        with Path.open(labels_path, "r") as file:
            for line in file:
                label_data = json.loads(line.strip())

                # If the line number is greater than the number of lines in the log file,
                # stop reading the labels file. This may happen if the log file was truncated.
                if label_data["line"] >= len(log_df):
                    break

                tactic, techniques = LABELS_TO_TTPS[str(label_data["labels"])]

                log_df.loc[log_df["line_number"] == label_data["line"], ["tactic", "techniques"]] = [
                    tactic,
                    "[" + ", ".join(techniques) + "]",
                ]

    return log_df


def import_data() -> None:
    """Import the dataset and process the logs."""
    print("\nProcessing logs...")

    for device in devices:
        print(f"  - {device}")

        # Go to the logs subdir of the current device
        logs_dir = Path(logs_subdir_path) / device / "logs"

        # Walk through the logs directory and read all log files
        for root, _, files in os.walk(logs_dir):
            # The log app corresponds to the relative path from the logs directory.
            # If the log is a system log, the log app will be empty.
            app = os.path.relpath(root, logs_dir) if root != logs_dir else ""

            for file_name in files:
                if "log" in file_name:
                    print(f"    > {file_name}...", end="")

                    # Binary files should be skipped.
                    # If the file is binary, an exception will be raised.
                    try:
                        log_df = read_log_and_labels(device, app, file_name)
                    except UnicodeDecodeError:
                        continue

                    if all(log_df["tactic"] == ""):
                        print("No tactics, skipping.")
                        continue

                    log_df = log_df[log_df["tactic"] != ""]

                    file_logs = []
                    for candidate_log in log_df.itertuples():
                        if any(
                            fuzz.ratio(candidate_log.text, existing_log.text) > SIMILARITY_THRESHOLD
                            for existing_log in file_logs
                        ):
                            continue
                        file_logs.append(candidate_log)

                    log_df = pd.DataFrame(file_logs)

                    Path.mkdir(Path(OUT_DIR), exist_ok=True, parents=True)

                    log_df.to_csv(
                        Path(OUT_DIR) / "out.csv",
                        index=False,
                        mode="a",
                    )

                    # Delete the dataframe to free up memory
                    del log_df

                    print("✔")

    print("Done.")

In [52]:
if Path(OUT_DIR).exists() and DELETE_OUT_IF_EXISTS:
    print("Deleting existing output directory...")
    shutil.rmtree(OUT_DIR)

if not Path(OUT_DIR).exists():
    import_data()
else:
    print("Output directory already exists. Skipping data import.")

Deleting existing output directory...

Processing logs...
  - morris_mail
    > mail.log.1...No tactics, skipping.
    > auth.log.1...No tactics, skipping.
    > syslog.1...No tactics, skipping.
    > syslog.3...No tactics, skipping.
    > mail.log...No tactics, skipping.
    > user.log...No tactics, skipping.
    > syslog...No tactics, skipping.
    > auth.log...No tactics, skipping.
    > syslog.4...No tactics, skipping.
    > syslog.2...No tactics, skipping.
    > user.log.1...No tactics, skipping.
    > mainlog.2...No tactics, skipping.
    > mainlog.4...No tactics, skipping.
    > mainlog.1...No tactics, skipping.
    > mainlog...No tactics, skipping.
    > mainlog.3...No tactics, skipping.
    > horde-access.log...No tactics, skipping.
    > horde-error.log...No tactics, skipping.
  - inet-dns
    > auth.log.1...No tactics, skipping.
    > syslog.1...No tactics, skipping.
    > syslog.3...No tactics, skipping.
    > dnsmasq.log...No tactics, skipping.
    > syslog...No tactics, s